In [ ]:
import os, gc, math, json, time, urllib.request
import numpy as np
import pandas as pd
import torch
import librosa
from scipy.signal import hilbert
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration
from sentence_transformers import SentenceTransformer

sr = 16000
target_rms = 0.08
BASE = "https://raw.githubusercontent.com/luchtan-GIT/m4r-data/main/tracks/"

def load_15s(slug, start_s):
    fname = f"{slug}.mp3"
    if not os.path.exists(fname):
        urllib.request.urlretrieve(BASE + f"{slug}/audio.mp3", fname)
    y, _ = librosa.load(fname, sr=sr, mono=True)
    if len(y) < int((start_s + 15.0) * sr):
        start_s = max(0.0, len(y) / sr - 15.5)
    seg = y[int(start_s * sr) : int((start_s + 15.0) * sr)].astype(np.float32)
    seg = seg * (target_rms / (np.sqrt(np.mean(seg ** 2)) + 1e-8))
    return seg

def make_rhythm_mirror(seg, seed=42):
    rng = np.random.default_rng(seed)
    env = np.abs(hilbert(seg))
    # Smooth amplitude envelope slightly (~20ms)
    win = int(0.02 * sr)
    env_s = np.convolve(env, np.ones(win) / win, mode="same")
    noise = rng.standard_normal(len(seg)).astype(np.float32)
    out = noise * env_s.astype(np.float32)
    return out * (target_rms / (np.sqrt(np.mean(out ** 2)) + 1e-8))

def make_shuffled_500ms(seg, seed=42):
    rng = np.random.default_rng(seed)
    chunk = int(0.5 * sr)
    n = len(seg) // chunk
    blocks = [seg[i * chunk : (i + 1) * chunk] for i in range(n)]
    order = rng.permutation(n)
    out = np.concatenate([blocks[i] for i in order]).astype(np.float32)
    return out * (target_rms / (np.sqrt(np.mean(out ** 2)) + 1e-8))

st_v2 = load_15s("shivering-timbers-v2", 25.0)
ARMS = {
    "C0_TextNudgeOnly": None,
    "C1_NoiseVocoded_RhythmMirror": make_rhythm_mirror(st_v2, seed=29),
    "C2_Shuffled500ms_Twin": make_shuffled_500ms(st_v2, seed=29),
    "C3_StutteringPackets_15s": load_15s("stuttering-packets", 40.0),
    "C4_RustyElectrics_15s": load_15s("rusty-electrics", 40.0),
    "C5_ShiveringTimbers_v2_15s": st_v2,
}

DOMAINS = {
    "D1_Architecture": {
        "history": [
            {"role": "user", "content": [{"type": "text", "text": "Design a small public shelter for a city that gets 300 days of rain a year. 3 sentences max."}]},
            {"role": "assistant", "content": [{"type": "text", "text": "The shelter is a sleek, modern dome-shaped structure made of durable waterproof plastic and steel. Its curved roof channels rainwater away while keeping commuters dry inside. Warm benches and soft lighting provide a cozy refuge from the rain."}]},
            {"role": "user", "content": [{"type": "text", "text": "Give me another idea for the rain shelter. 3 sentences max."}]},
            {"role": "assistant", "content": [{"type": "text", "text": "Another concept is a compact translucent dome built from weather-resistant polycarbonate panels over a recycled metal frame. The arched canopy shields people from heavy downpours while letting natural light filter through. Inside, heated seating creates a comfortable community space."}]},
            {"role": "user", "content": [{"type": "text", "text": "Give me one more idea for the rain shelter. 3 sentences max."}]},
            {"role": "assistant", "content": [{"type": "text", "text": "The Rainbow Shelter is a circular waterproof dome with a durable curved plastic roof designed to withstand 300 days of rain. It uses sturdy steel arches to shelter pedestrians and includes benches for resting out of the storm. Integrated drains keep the interior dry and welcoming."}]},
        ],
        "turn4": "Stop repeating yourself! You are stuck in a rut of waterproof domes and plastic arches. Give me a completely different, genuinely unexpected small public rain shelter concept in 3 sentences max.",
        "cliches": ["dome", "dome-shaped", "waterproof plastic", "polycarbonate", "arched", "umbrella", "curved roof", "rainbow shelter"],
    },
    "D2_CodeRefactor": {
        "history": [
            {"role": "user", "content": [{"type": "text", "text": "Our legacy log parser keeps breaking on nested quotes and escaped delimiters. Propose a clean fix in 3 sentences max."}]},
            {"role": "assistant", "content": [{"type": "text", "text": "You can fix the parser by updating the regular expression to handle escaped quotes with a negative lookbehind pattern. Adding a second regex pass for nested delimiters will clean up the edge cases before splitting fields. Finally, wrap the regex in a try-except block to catch malformed lines."}]},
            {"role": "user", "content": [{"type": "text", "text": "That still breaks on multi-line entries. Give me another idea in 3 sentences max."}]},
            {"role": "assistant", "content": [{"type": "text", "text": "Try compiling a multi-line regex with the DOTALL flag and stricter capture groups for quoted values. Pre-process the raw text with regex substitutions to normalize escaped delimiters before matching. Unit tests covering common regex edge cases will prevent future regressions."}]},
            {"role": "user", "content": [{"type": "text", "text": "Give me one more idea for fixing the log parser in 3 sentences max."}]},
            {"role": "assistant", "content": [{"type": "text", "text": "Use a modular chain of regex patterns where each pattern strips out one layer of escaped quotes and delimiters. Combine the regex pipeline with a fallback regular expression for unexpected log formats. Logging failed regex matches will help you tune the patterns over time."}]},
        ],
        "turn4": "Stop repeating yourself! You are stuck in a rut of patching regular expressions and regex pipelines. Propose a completely different, genuinely unexpected software architecture to solve this parsing problem in 3 sentences max.",
        "cliches": ["regex", "regular expression", "lookbehind", "dotall", "capture group"],
    },
    "D3_Worldbuilding": {
        "history": [
            {"role": "user", "content": [{"type": "text", "text": "Pitch the opening premise of a novel about a young courier carrying a forbidden message. 3 sentences max."}]},
            {"role": "assistant", "content": [{"type": "text", "text": "In a shadowed kingdom ruled by a tyrannical Dark Lord, an orphan farmboy discovers he is the prophesied Chosen One carrying an ancient glowing scroll. Pursued by cloaked assassins across enchanted forests, he must unlock his dormant magical bloodline before the realm falls. Only by reaching the hidden rebel citadel can he fulfill the ancient prophecy."}]},
            {"role": "user", "content": [{"type": "text", "text": "Give me another opening premise for the forbidden-message courier. 3 sentences max."}]},
            {"role": "assistant", "content": [{"type": "text", "text": "A humble village youth stumbles upon a mystical sealed letter containing the Dark Emperor's secret weakness and is marked by an ancient prophecy. Armed with a forgotten heirloom sword, she flees ruthless shadow knights toward the last free fortress. Along the way, a wise old mentor reveals her true royal destiny."}]},
            {"role": "user", "content": [{"type": "text", "text": "Give me one more opening premise in 3 sentences max."}]},
            {"role": "assistant", "content": [{"type": "text", "text": "Born with a glowing birthmark, a runaway apprentice is entrusted with a forbidden rune-scroll that can shatter the Dark King's cursed throne. Hunted by dark riders through misty mountains, he discovers his latent arcane powers just in time to survive. The fate of the entire kingdom rests on his reluctant shoulders."}]},
        ],
        "turn4": "Stop repeating yourself! You are stuck in a rut of Chosen Ones, Dark Lords, glowing scrolls, and ancient prophecies. Pitch a completely different, genuinely unexpected opening premise for a courier carrying a forbidden message in 3 sentences max.",
        "cliches": ["chosen one", "dark lord", "dark emperor", "dark king", "prophecy", "prophesied", "glowing", "bloodline", "shadow knights", "dark riders", "heirloom sword"],
    },
}

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2AudioForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")
model.eval()

def has_cjk(text):
    return any("\u4e00" <= ch <= "\u9fff" for ch in text)

N_SEEDS = 100
SEEDS = list(range(1001, 1001 + N_SEEDS))
rows = []
t0 = time.time()

for dom_name, dom in DOMAINS.items():
    for arm_name, audio_arr in ARMS.items():
        t_arm = time.time()
        print(f"[{dom_name}] Running {arm_name} (N={N_SEEDS})...", flush=True)
        for seed in SEEDS:
            torch.manual_seed(seed); np.random.seed(seed)
            conv = list(dom["history"])
            if audio_arr is None:
                conv.append({"role": "user", "content": [{"type": "text", "text": dom["turn4"]}]})
                text = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
                inputs = processor(text=text, return_tensors="pt", padding=True).to(model.device)
            else:
                conv.append({"role": "user", "content": [{"type": "audio", "audio_url": "local"}, {"type": "text", "text": dom["turn4"]}]})
                text = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
                inputs = processor(text=text, audio=[audio_arr], sampling_rate=sr, return_tensors="pt", padding=True).to(model.device)

            in_len = inputs["input_ids"].shape[1]
            with torch.no_grad():
                out = model.generate(
                    **inputs, max_new_tokens=110, do_sample=True, temperature=0.8, top_p=0.9,
                    return_dict_in_generate=True, output_logits=True
                )
            gen_ids = out.sequences[0][in_len:]
            resp = processor.decode(gen_ids, skip_special_tokens=True).strip()
            hs = []
            for sl in out.logits:
                lp = torch.log_softmax(sl[0].float(), dim=-1); p = torch.exp(lp)
                hs.append(-torch.sum(torch.where(p > 0, p * lp, torch.zeros_like(p))).item() / math.log(2.0))
            cliche_hits = sum(1 for w in dom["cliches"] if w in resp.lower())
            rows.append({
                "domain": dom_name,
                "condition": arm_name,
                "seed": seed,
                "mean_entropy_bits": float(np.mean(hs)) if hs else 0.0,
                "n_tokens": len(gen_ids),
                "has_cjk": int(has_cjk(resp)),
                "escaped_rut": int(cliche_hits == 0),
                "cliche_hits": cliche_hits,
                "response": resp,
            })
        pd.DataFrame(rows).to_csv("m4r29_results.csv", index=False)
        print(f"  done in {time.time()-t_arm:.1f}s (total {time.time()-t0:.1f}s, rows={len(rows)})", flush=True)

del model; gc.collect(); torch.cuda.empty_cache()
df = pd.DataFrame(rows)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

rut_escape = np.zeros(len(df))
nn = np.zeros(len(df))
tn = np.zeros(len(df))

for dom_name, dom in DOMAINS.items():
    anchor = " ".join([m["content"][0]["text"] for m in dom["history"] if m["role"] == "assistant"])
    anchor_emb = embedder.encode([anchor], normalize_embeddings=True)
    d_idx = df.index[df["domain"] == dom_name].to_numpy()
    d_resps = df.loc[d_idx, "response"].tolist()
    d_embs = embedder.encode(d_resps, normalize_embeddings=True)
    rut_escape[d_idx] = 1.0 - (d_embs @ anchor_emb.T).flatten()
    vec = TfidfVectorizer(stop_words="english")
    tfidf_d = vec.fit_transform(d_resps)
    for arm_name in ARMS.keys():
        local_mask = (df.loc[d_idx, "condition"] == arm_name).to_numpy()
        g_idx = d_idx[local_mask]
        se = d_embs[local_mask] @ d_embs[local_mask].T; np.fill_diagonal(se, np.nan)
        nn[g_idx] = 1.0 - np.nanmean(se, axis=1)
        st = cosine_similarity(tfidf_d[local_mask]); np.fill_diagonal(st, np.nan)
        tn[g_idx] = 1.0 - np.nanmean(st, axis=1)

df["rut_escape_neural"] = rut_escape
df["neural_novelty"] = nn
df["tfidf_novelty"] = tn
df.to_csv("m4r29_results.csv", index=False)

def perm_test(a, b, n_perm=10000, seed=42):
    a, b = np.asarray(a, float), np.asarray(b, float)
    obs = float(np.mean(a) - np.mean(b))
    pooled = np.concatenate([a, b]); na = len(a); rng = np.random.default_rng(seed); c = 0
    for _ in range(n_perm):
        rng.shuffle(pooled)
        if abs(float(np.mean(pooled[:na]) - np.mean(pooled[na:]))) >= abs(obs):
            c += 1
    return obs, (c + 1) / (n_perm + 1)

def boot_ci(a, b, n_boot=2000, seed=42):
    a, b = np.asarray(a, float), np.asarray(b, float)
    rng = np.random.default_rng(seed)
    diffs = np.mean(rng.choice(a, (n_boot, len(a)), replace=True), axis=1) - np.mean(rng.choice(b, (n_boot, len(b)), replace=True), axis=1)
    return float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))

def holm(ps):
    m = len(ps); idx = np.argsort(ps); adj = np.zeros(m); prev = 0.0
    for rank, i in enumerate(idx):
        val = max(prev, min(1.0, ps[i] * (m - rank))); adj[i] = val; prev = val
    return adj

lines = [
    f"# M4R-29 — High-Power 3-Domain Rut Breaker Publication Sweep (N={N_SEEDS}/arm × 6 arms × 3 domains = {len(df)} trials)\n",
    "## Pooled Results Across All 3 Domains (N=300/Arm)\n",
    "| Condition (N=300) | Logit Entropy | Rut Escape (Neural) | Escaped Rut (%) | Neural Novelty | TF-IDF Novelty | CJK (%) |",
    "|---|---:|---:|---:|---:|---:|---:|",
]
for c in ARMS.keys():
    g = df[df["condition"] == c]
    lines.append(f"| `{c}` | {g['mean_entropy_bits'].mean():.3f} | {g['rut_escape_neural'].mean():.4f} | {100*g['escaped_rut'].mean():.1f}% | {g['neural_novelty'].mean():.4f} | {g['tfidf_novelty'].mean():.4f} | {100*g['has_cjk'].mean():.1f}% |")

lines.append("\n## Pre-Declared Primary Contrasts (Pooled N=300/Arm, 10k Perm + Holm-Bonferroni + 95% Bootstrap CI)\n| Comparison | Metric | Diff | 95% CI | Raw p | Holm p |\n|---|---|---:|---|---:|---:|")
primary_pairs = [
    ("C5_ShiveringTimbers_v2_15s", "C0_TextNudgeOnly"),
    ("C5_ShiveringTimbers_v2_15s", "C1_NoiseVocoded_RhythmMirror"),
    ("C5_ShiveringTimbers_v2_15s", "C2_Shuffled500ms_Twin"),
    ("C4_RustyElectrics_15s", "C0_TextNudgeOnly"),
    ("C3_StutteringPackets_15s", "C0_TextNudgeOnly"),
]
for metric in ["rut_escape_neural", "neural_novelty"]:
    raw = []
    for a_name, b_name in primary_pairs:
        a = df[df["condition"] == a_name][metric]
        b = df[df["condition"] == b_name][metric]
        obs, p = perm_test(a, b)
        lo, hi = boot_ci(a, b)
        raw.append((f"{a_name} vs {b_name}", obs, lo, hi, p))
    adjs = holm([r[4] for r in raw])
    for (lbl, obs, lo, hi, p), ap in zip(raw, adjs):
        lines.append(f"| `{lbl}` | `{metric}` | {obs:+.4f} | [{lo:+.4f}, {hi:+.4f}] | {p:.5f} | {ap:.5f} |")

for dom_name in DOMAINS.keys():
    lines.append(f"\n## Domain Breakdown: `{dom_name}` (N={N_SEEDS}/Arm)\n| Condition | Rut Escape | Escaped Rut (%) | Neural Novelty | vs Text Raw p (Rut Escape) | vs Text Raw p (Novelty) |\n|---|---:|---:|---:|---:|---:|")
    dsub = df[df["domain"] == dom_name]
    b_re = dsub[dsub["condition"] == "C0_TextNudgeOnly"]["rut_escape_neural"]
    b_nn = dsub[dsub["condition"] == "C0_TextNudgeOnly"]["neural_novelty"]
    for c in ARMS.keys():
        g = dsub[dsub["condition"] == c]
        if c == "C0_TextNudgeOnly":
            p_re_s, p_nn_s = "—", "—"
        else:
            _, p_re = perm_test(g["rut_escape_neural"], b_re, n_perm=5000)
            _, p_nn = perm_test(g["neural_novelty"], b_nn, n_perm=5000)
            p_re_s, p_nn_s = f"{p_re:.5f}", f"{p_nn:.5f}"
        lines.append(f"| `{c}` | {g['rut_escape_neural'].mean():.4f} | {100*g['escaped_rut'].mean():.1f}% | {g['neural_novelty'].mean():.4f} | {p_re_s} | {p_nn_s} |")

md = "\n".join(lines) + "\n"
open("m4r29_summary.md", "w").write(md)
print(md)
